### Download sample videos off of youtube for testing actual video inference

In [2]:
import os
from pathlib import Path

NOTEBOOK_DIR = os.getcwd()
PROJECT_DIR  = Path(NOTEBOOK_DIR) / "experiments"
VIDEO_DIR    = PROJECT_DIR / "example_videos"

US_DIR = VIDEO_DIR / "us"
EU_DIR = VIDEO_DIR / "eu"

VIDEO_DIR.mkdir(parents=True, exist_ok=True)
US_DIR.mkdir(parents=True, exist_ok=True)
EU_DIR.mkdir(parents=True, exist_ok=True)

# US Videos
!yt-dlp --download-sections "*0:00-2:00" -f "best[height<=720]" "https://www.youtube.com/watch?v=pwbomtuwYsI" -o "{US_DIR}/%(title)s.%(ext)s"
!yt-dlp --download-sections "*0:10-2:00" -f "best[height<=720]" "https://www.youtube.com/watch?v=W5GW_Lb9I2A" -o "{US_DIR}/%(title)s.%(ext)s"
!yt-dlp --download-sections "*1:10-3:50" -f "best[height<=720]" "https://www.youtube.com/watch?v=EfntgCSet1o" -o "{US_DIR}/%(title)s.%(ext)s"

# EU Videos
!yt-dlp --download-sections "*1:50-10:00" -f "best[height<=720]" "https://www.youtube.com/watch?v=GarVk7hCHNg" -o "{EU_DIR}/%(title)s.%(ext)s"

^C
[youtube] Extracting URL: https://www.youtube.com/watch?v=pwbomtuwYsI
[youtube] pwbomtuwYsI: Downloading webpage
[youtube] pwbomtuwYsI: Downloading android vr player API JSON
[info] pwbomtuwYsI: Downloading 1 format(s): 18
[info] pwbomtuwYsI: Downloading 1 time ranges: 0.0-120.0
[download] c:\Projects\TrafficSignRecognition\model_training\experiments\example_videos\us\Driving Through Fort Wayne, Indiana.mp4 has already been downloaded

[download] 100% of    9.81MiB


[youtube] Extracting URL: https://www.youtube.com/watch?v=W5GW_Lb9I2A
[youtube] W5GW_Lb9I2A: Downloading webpage
[youtube] W5GW_Lb9I2A: Downloading android vr player API JSON
[info] W5GW_Lb9I2A: Downloading 1 format(s): 18
[info] W5GW_Lb9I2A: Downloading 1 time ranges: 10.0-120.0
[download] c:\Projects\TrafficSignRecognition\model_training\experiments\example_videos\us\New york snow 2 am night.mp4 has already been downloaded

[download] 100% of    4.98MiB


[youtube] Extracting URL: https://www.youtube.com/watch?v=EfntgCSet1o
[youtube] EfntgCSet1o: Downloading webpage
[youtube] EfntgCSet1o: Downloading android vr player API JSON
[info] EfntgCSet1o: Downloading 1 format(s): 18
[info] EfntgCSet1o: Downloading 1 time ranges: 70.0-230.0
[download] c:\Projects\TrafficSignRecognition\model_training\experiments\example_videos\us\08 CVPI Dashcam (Tornado⧸Flash Flood Warning).mp4 has already been downloaded

[download] 100% of   12.49MiB


[youtube] Extracting URL: https://www.youtube.com/watch?v=GarVk7hCHNg
[youtube] GarVk7hCHNg: Downloading webpage
[youtube] GarVk7hCHNg: Downloading android vr player API JSON
[info] GarVk7hCHNg: Downloading 1 format(s): 18
[info] GarVk7hCHNg: Downloading 1 time ranges: 110.0-600.0
[download] c:\Projects\TrafficSignRecognition\model_training\experiments\example_videos\eu\Driving in Munich, Germany ｜ Dashcam City Tour 4K.mp4 has already been downloaded

[download] 100% of   45.12MiB


### Convert .mov to .mp4

In [7]:
%pip install moviepy

   ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.0 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/7.0 MB 2.4 MB/s eta 0:00:03
   --------- ------------------------------ 1.6/7.0 MB 3.1 MB/s eta 0:00:02
   ------------- -------------------------- 2.4/7.0 MB 3.3 MB/s eta 0:00:02
   ---------------- ----------------------- 2.9/7.0 MB 3.4 MB/s eta 0:00:02
   --------------------- ------------------ 3.7/7.0 MB 3.4 MB/s eta 0:00:01
   ------------------------- -------------- 4.5/7.0 MB 3.4 MB/s eta 0:00:01
   ------------------------------ --------- 5.2/7.0 MB 3.5 MB/s eta 0:00:01
   ---------------------------------- ----- 6.0/7.0 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------  6.8/7.0 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 7.0/7.0 MB 3.5 MB/s  0:00:02

  Attempting uninstall: pillow

    Found existing installation: pillow 12.1.1

    Uninstalling pillow-

  You can safely remove it manually.


In [3]:
from moviepy import VideoFileClip

# Define file paths
input_path = "experiments/example_videos/us/My_Dash3.MOV"
output_path = "experiments/example_videos/us/My_Dash3.mp4"

# Load, convert, and save
clip = VideoFileClip(input_path)
clip.write_videofile(output_path, codec='libx264')

MoviePy - Building video experiments/example_videos/us/My_Dash3.mp4.
MoviePy - Writing audio in My_Dash3TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video experiments/example_videos/us/My_Dash3.mp4



MoviePy - Done !
MoviePy - video ready experiments/example_videos/us/My_Dash3.mp4


# Run inference on videos

In [1]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from collections import defaultdict


# ── Tracker ───────────────────────────────────────────────────────────────────

class DetectorTracker:
    """
    Lightweight tracker that stabilizes a YOLO detector's per-frame predictions.
    No classifier — uses the detector's own class/confidence outputs directly.

    Locking behaviour:
      - A track accumulates per-class confidence votes across frames.
      - Once it has enough hits AND majority-vote confidence exceeds
        lock_conf_threshold, the label+conf are frozen until the track dies.
    """

    def __init__(
        self,
        iou_threshold       = 0.15,
        lock_conf_threshold = 0.70,
        max_frames_missing  = 15,
        min_hits_to_lock    = 3,
    ):
        self.iou_threshold       = iou_threshold
        self.lock_conf_threshold = lock_conf_threshold
        self.max_frames_missing  = max_frames_missing
        self.min_hits_to_lock    = min_hits_to_lock
        self.tracks              = []
        self.next_id             = 0

    # ── Geometry helpers ──────────────────────────────────────────────────────

    def _iou(self, a, b):
        xA = max(a[0], b[0]); yA = max(a[1], b[1])
        xB = min(a[2], b[2]); yB = min(a[3], b[3])
        inter = max(0, xB - xA) * max(0, yB - yA)
        if inter == 0:
            return 0.0
        return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)

    def _center_dist(self, a, b):
        ca = ((a[0]+a[2])/2, (a[1]+a[3])/2)
        cb = ((b[0]+b[2])/2, (b[1]+b[3])/2)
        return ((ca[0]-cb[0])**2 + (ca[1]-cb[1])**2) ** 0.5

    def _matches(self, box, track):
        iou  = self._iou(box, track["box"])
        dist = self._center_dist(box, track["box"])
        diag = ((track["box"][2]-track["box"][0])**2 +
                (track["box"][3]-track["box"][1])**2) ** 0.5
        return iou >= self.iou_threshold or dist < diag * 1.5, iou

    # ── Majority-vote label from history ─────────────────────────────────────

    def _majority_label(self, history):
        counts = defaultdict(float)
        for label, conf in history:
            counts[label] += conf          # weight by confidence
        return max(counts, key=counts.get)

    # ── Main update ───────────────────────────────────────────────────────────

    def update(self, detections):
        """
        detections: list of (box_xyxy, label_str, conf_float)

        Returns list of (box_xyxy, label_str, conf_float, is_locked)
        """
        for t in self.tracks:
            t["frames_missing"] += 1

        used = set()
        output = []

        for box, label, conf in detections:
            best_iou   = -1
            best_track = None

            for t in self.tracks:
                if t["id"] in used:
                    continue
                matched, iou = self._matches(box, t)
                if matched and iou > best_iou:
                    best_iou   = iou
                    best_track = t

            if best_track is not None:
                best_track["box"]            = box
                best_track["frames_missing"] = 0
                best_track["hits"]          += 1
                used.add(best_track["id"])

                if best_track["locked"]:
                    # Frozen — emit stored label unchanged
                    output.append((box, best_track["label"],
                                   best_track["conf"], True))
                else:
                    # Accumulate history, vote
                    best_track["history"].append((label, conf))
                    if len(best_track["history"]) > 8:
                        best_track["history"].pop(0)

                    stable_label = self._majority_label(best_track["history"])
                    best_conf    = max(
                        c for l, c in best_track["history"]
                        if l == stable_label
                    )
                    best_track["label"] = stable_label
                    best_track["conf"]  = best_conf

                    if (best_track["hits"] >= self.min_hits_to_lock
                            and best_conf >= self.lock_conf_threshold):
                        best_track["locked"] = True

                    output.append((box, stable_label, best_conf,
                                   best_track["locked"]))
            else:
                # New track
                track = {
                    "id"            : self.next_id,
                    "box"           : box,
                    "label"         : label,
                    "conf"          : conf,
                    "locked"        : False,
                    "hits"          : 1,
                    "frames_missing": 0,
                    "history"       : [(label, conf)],
                }
                self.tracks.append(track)
                self.next_id += 1
                output.append((box, label, conf, False))

        # Expire stale tracks
        self.tracks = [
            t for t in self.tracks
            if t["frames_missing"] <= self.max_frames_missing
        ]

        return output


# ── Per-label colour (deterministic) ─────────────────────────────────────────

_label_colors: dict = {}

def _get_color(label: str):
    if label not in _label_colors:
        rng   = np.random.default_rng(abs(hash(label)) % (2**32))
        color = rng.integers(100, 230, size=3).tolist()
        _label_colors[label] = tuple(color)
    return _label_colors[label]


# ── Single-video inference ────────────────────────────────────────────────────

def run_video_inference(
    video_path,
    detector_path       = "experiments/detect/baseline_v1/weights/best.pt",
    output_dir          = "experiments/example_videos/output",
    conf_threshold      = 0.40,          # detector confidence gate
    iou_threshold       = 0.45,          # NMS IoU threshold
    imgsz               = 640,
    frame_skip          = 1,             # process every N-th frame; draw last on skipped
    lock_conf_threshold = 0.70,          # confidence to freeze a track's label
    max_frames_missing  = 15,            # frames before a track is dropped
    min_hits_to_lock    = 3,             # detections needed before locking
    min_box_area_frac   = 0.0003,        # drop boxes smaller than this fraction of frame
    device              = "cuda",
    show_progress       = True,
):
    video_path = Path(video_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    out_path = output_dir / f"{video_path.stem}_annotated.mp4"

    # ── Load detector ─────────────────────────────────────────────────────────
    detector = YOLO(detector_path)
    detector.to(device)

    # ── Open video ────────────────────────────────────────────────────────────
    cap          = cv2.VideoCapture(str(video_path))
    fps          = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    writer = cv2.VideoWriter(
        str(out_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    print(f"  Video   : {video_path.name}")
    print(f"  Size    : {width}x{height} @ {fps:.1f} fps")
    print(f"  Frames  : {total_frames:,}  (every {frame_skip})")
    print(f"  Output  : {out_path}\n")

    tracker = DetectorTracker(
        lock_conf_threshold = lock_conf_threshold,
        max_frames_missing  = max_frames_missing,
        min_hits_to_lock    = min_hits_to_lock,
    )

    class_names      = detector.names          # {int: str}
    stats            = defaultdict(int)
    label_counts     = defaultdict(int)
    last_annotations = []
    frame_idx        = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1

        if frame_idx % frame_skip == 0:
            # ── Run detector ──────────────────────────────────────────────
            result    = detector(
                frame,
                imgsz   = imgsz,
                conf    = conf_threshold,
                iou     = iou_threshold,
                verbose = False,
            )[0]

            # ── Parse detections ──────────────────────────────────────────
            min_area = width * height * min_box_area_frac
            dets = []
            if len(result.boxes):
                boxes  = result.boxes.xyxy.cpu().numpy()
                confs  = result.boxes.conf.cpu().numpy()
                cls_ids = result.boxes.cls.cpu().numpy().astype(int)
                for box, conf, cls_id in zip(boxes, confs, cls_ids):
                    if (box[2]-box[0]) * (box[3]-box[1]) < min_area:
                        continue
                    label = class_names.get(cls_id, str(cls_id))
                    dets.append((box, label, float(conf)))

            # ── Update tracker ────────────────────────────────────────────
            annotations      = tracker.update(dets)
            last_annotations = annotations

            for _, label, _, _ in annotations:
                label_counts[label] += 1
            stats["frames_processed"] += 1
            stats["total_detections"] += len(annotations)
        else:
            annotations = last_annotations

        # ── Draw ──────────────────────────────────────────────────────────
        for box, label, conf, locked in annotations:
            x1, y1, x2, y2 = map(int, box)
            color = _get_color(label)

            # Slightly thicker white border for locked tracks
            if locked:
                cv2.rectangle(frame, (x1-2, y1-2), (x2+2, y2+2),
                              (255, 255, 255), 1)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            prefix       = "[L] " if locked else ""
            display_text = f"{prefix}{label}  {conf:.2f}"
            font         = cv2.FONT_HERSHEY_SIMPLEX
            font_scale   = 0.42
            thickness    = 1
            (tw, th), _  = cv2.getTextSize(display_text, font, font_scale, thickness)
            text_y       = max(y1 - 4, th + 6)
            cv2.rectangle(frame,
                          (x1, text_y - th - 4),
                          (x1 + tw + 6, text_y + 2),
                          color, -1)
            cv2.putText(frame, display_text, (x1 + 3, text_y - 2),
                        font, font_scale, (0, 0, 0), thickness, cv2.LINE_AA)

        # Frame counter overlay
        cv2.putText(
            frame,
            f"frame {frame_idx}  |  {len(annotations)} sign(s)",
            (10, 26), cv2.FONT_HERSHEY_SIMPLEX,
            0.55, (255, 255, 255), 1, cv2.LINE_AA,
        )

        writer.write(frame)

        if show_progress and frame_idx % 200 == 0:
            pct = 100 * frame_idx / max(total_frames, 1)
            print(f"    {frame_idx:>6} / {total_frames}  ({pct:.1f}%)")

    cap.release()
    writer.release()

    # ── Summary ───────────────────────────────────────────────────────────────
    proc = stats["frames_processed"]
    print(f"\n  ── Done {'─'*50}")
    print(f"  Frames processed : {proc:,}")
    print(f"  Total detections : {stats['total_detections']:,}")
    print(f"  Avg per frame    : {stats['total_detections'] / max(proc, 1):.2f}")
    print(f"  Output           : {out_path}")

    if label_counts:
        print(f"\n  {'Label':<45} {'Count':>6}")
        print(f"  {'─'*45} {'─'*6}")
        for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
            print(f"  {label:<45} {count:>6,}")

    return out_path


# ── Batch runner ──────────────────────────────────────────────────────────────

def run_all_videos(
    video_dir     = "experiments/example_videos",
    detector_path = "experiments/detect/baseline_v1/weights/best.pt",
    output_dir    = "experiments/example_videos/output",
    **kwargs,
):
    """Run inference on every .mp4 / .webm in video_dir."""
    video_dir = Path(video_dir)
    videos    = sorted(video_dir.glob("*.mp4")) + sorted(video_dir.glob("*.webm"))

    if not videos:
        print(f"No videos found in {video_dir}")
        return {}

    print(f"Found {len(videos)} video(s) in {video_dir}\n")
    results = {}

    for video_path in videos:
        print(f"{'='*60}")
        print(f"  {video_path.name}")
        print(f"{'='*60}")
        out = run_video_inference(
            video_path    = video_path,
            detector_path = detector_path,
            output_dir    = output_dir,
            **kwargs,
        )
        results[video_path.name] = out
        print()

    return results

In [4]:
# Single video
run_video_inference(
    video_path    = "experiments/example_videos/us/My_Dash3.mp4",
    detector_path = "experiments/baseline_v1_1280/weights/best.pt",
    conf_threshold    = 0.15,
    iou_threshold     = 0.45,
    imgsz             = 1280,
    min_box_area_frac = 0.0001,
    min_hits_to_lock  = 2,
    lock_conf_threshold = 0.60,
)

  Video   : My_Dash3.mp4
  Size    : 1920x1080 @ 30.0 fps
  Frames  : 13,018  (every 1)
  Output  : experiments\example_videos\output\My_Dash3_annotated.mp4

       200 / 13018  (1.5%)
       400 / 13018  (3.1%)
       600 / 13018  (4.6%)
       800 / 13018  (6.1%)
      1000 / 13018  (7.7%)
      1200 / 13018  (9.2%)
      1400 / 13018  (10.8%)
      1600 / 13018  (12.3%)
      1800 / 13018  (13.8%)
      2000 / 13018  (15.4%)
      2200 / 13018  (16.9%)
      2400 / 13018  (18.4%)
      2600 / 13018  (20.0%)
      2800 / 13018  (21.5%)
      3000 / 13018  (23.0%)
      3200 / 13018  (24.6%)
      3400 / 13018  (26.1%)
      3600 / 13018  (27.7%)
      3800 / 13018  (29.2%)
      4000 / 13018  (30.7%)
      4200 / 13018  (32.3%)
      4400 / 13018  (33.8%)
      4600 / 13018  (35.3%)
      4800 / 13018  (36.9%)
      5000 / 13018  (38.4%)
      5200 / 13018  (39.9%)
      5400 / 13018  (41.5%)
      5600 / 13018  (43.0%)
      5800 / 13018  (44.6%)
      6000 / 13018  (46.1%)
      62

WindowsPath('experiments/example_videos/output/My_Dash3_annotated.mp4')

In [ ]:
# All videos in a folder
run_all_videos(
    video_dir         = "experiments/example_videos/us",
    detector_path     = "experiments/baseline_v1_1280/weights/best.pt",
    conf_threshold    = 0.15,
    iou_threshold     = 0.45,
    imgsz             = 1280,
    min_box_area_frac = 0.0001,
    min_hits_to_lock  = 2,
    lock_conf_threshold = 0.60,
)

Found 6 video(s) in experiments\example_videos\us

  08 CVPI Dashcam (Tornado⧸Flash Flood Warning).mp4
  Video   : 08 CVPI Dashcam (Tornado⧸Flash Flood Warning).mp4
  Size    : 640x360 @ 30.0 fps
  Frames  : 4,821  (every 1)
  Output  : experiments\example_videos\output\08 CVPI Dashcam (Tornado⧸Flash Flood Warning)_annotated.mp4

       200 / 4821  (4.1%)
       400 / 4821  (8.3%)
       600 / 4821  (12.4%)
       800 / 4821  (16.6%)
      1000 / 4821  (20.7%)
      1200 / 4821  (24.9%)
      1400 / 4821  (29.0%)
      1600 / 4821  (33.2%)
      1800 / 4821  (37.3%)
      2000 / 4821  (41.5%)
      2200 / 4821  (45.6%)
      2400 / 4821  (49.8%)
      2600 / 4821  (53.9%)
      2800 / 4821  (58.1%)
      3000 / 4821  (62.2%)
      3200 / 4821  (66.4%)
      3400 / 4821  (70.5%)
      3600 / 4821  (74.7%)
      3800 / 4821  (78.8%)
      4000 / 4821  (83.0%)
      4200 / 4821  (87.1%)
      4400 / 4821  (91.3%)
      4600 / 4821  (95.4%)
      4800 / 4821  (99.6%)

  ── Done ───────────

{'08 CVPI Dashcam (Tornado⧸Flash Flood Warning).mp4': WindowsPath('experiments/example_videos/output/08 CVPI Dashcam (Tornado⧸Flash Flood Warning)_annotated.mp4'),
 'Driving Through Fort Wayne, Indiana.mp4': WindowsPath('experiments/example_videos/output/Driving Through Fort Wayne, Indiana_annotated.mp4'),
 'My_Dash.mp4': WindowsPath('experiments/example_videos/output/My_Dash_annotated.mp4'),
 'My_Dash2.mp4': WindowsPath('experiments/example_videos/output/My_Dash2_annotated.mp4'),
 'My_Dash3.mp4': WindowsPath('experiments/example_videos/output/My_Dash3_annotated.mp4'),
 'New york snow 2 am night.mp4': WindowsPath('experiments/example_videos/output/New york snow 2 am night_annotated.mp4')}